In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5),(0.5))
])

In [ ]:
train_data = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transforms
)

test_data = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transforms
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 10.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 169kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.06MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.48MB/s]


In [ ]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
from torch.nn.modules.linear import Linear
from torch.nn.modules.activation import ReLU
class KiyimCNN(nn.Module):
    def __init__(self):
      super(KiyimCNN, self).__init__()

      self.net = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2,),
        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Flatten(),
        nn.Linear(32*7*7, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )

    def forward(self, x):
       return self.net(x)


In [ ]:
model = KiyimCNN().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epoxa = 10

for epoch in range(epoxa):
  model.train()
  total_loss = 0
  for data, target in train_loader:
    data, target = data.to(device), target.to(device)

    output = model(data)
    loss = criterion(output, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  print(f"Epoxa: {epoch+1}, O'rtacha yo'qotish: {total_loss/len(train_loader)}")

Epoxa: 1, O'rtacha yo'qotish: 0.10230125360302628
Epoxa: 2, O'rtacha yo'qotish: 0.08879884560741484
Epoxa: 3, O'rtacha yo'qotish: 0.07976950256712735
Epoxa: 4, O'rtacha yo'qotish: 0.07067500401115977
Epoxa: 5, O'rtacha yo'qotish: 0.061760452230480344
Epoxa: 6, O'rtacha yo'qotish: 0.057610422698928354
Epoxa: 7, O'rtacha yo'qotish: 0.05160433549660568
Epoxa: 8, O'rtacha yo'qotish: 0.04742960152547651
Epoxa: 9, O'rtacha yo'qotish: 0.04131573066669516
Epoxa: 10, O'rtacha yo'qotish: 0.03700957409765106


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
  for data, target in test_loader:
    data, target = data.to(device), target.to(device)
    output = model(data)

    _, predicted = torch.max(output.data, 1)
    total += target.size(0)
    correct += (predicted == target).sum().item()

print(f"Test Accuracy: {100*correct/total}%")
#

Test Accuracy: 91.04%
